<style>
  .capa{max-width:960px;margin:16px auto;padding:28px;border:1px solid #e5e7eb;border-radius:12px;font-family:system-ui,-apple-system,"Segoe UI",Roboto,Arial,sans-serif;color:#1f2937}
  .topo{display:flex;gap:16px;align-items:flex-start}
  .logo{width:120px;height:auto;border:1px solid #e5e7eb;border-radius:6px;padding:6px;background:#fff}
  h1{margin:0 0 4px 0;font-size:22px;color:#111827;line-height:1.35}
  .sub{color:#6b7280;margin-bottom:10px;font-size:13px}
  .meta{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:12px}
  .pill{font-size:12px;padding:5px 10px;border:1px solid #e5e7eb;border-radius:999px;background:#f3f4f6}
  hr{border:none;border-top:1px solid #e5e7eb;margin:16px 0}
  h2{font-size:15px;margin:14px 0 6px 0;color:#111827}
  ul{margin:4px 0 0 18px}
  li{margin-bottom:5px;font-size:13.5px;line-height:1.5}
  p{font-size:13.5px;line-height:1.6;margin:6px 0}
</style>

<div class="capa" role="region" aria-label="Capa TCC UFMG">
  <div class="topo">
    <img class="logo" src="logo_ufmg.png" alt="Logo UFMG">
    <div>
      <h1>Modelagem de Preços de Combustíveis no Brasil (ANP, 2021–2025):<br>Avaliação e Seleção de Modelos de Regressão</h1>
      <div class="sub">Universidade Federal de Minas Gerais — Trabalho de Conclusão de Curso</div>
      <div class="meta">
        <span class="pill">Autoria: <b>Marina Ferreira</b></span>
        <span class="pill">Orientação: <b>Profa. Dra. Jussiane</b></span>
        <span class="pill">Versão 2.0</span>
        <span class="pill">Dados: ANP 2021–2025</span>
        <span class="pill">Última revisão: maio/2026</span>
      </div>
    </div>
  </div>

  <hr>

  <h2>Introdução e objetivo</h2>
  <p>
    Este caderno documenta o <em>pipeline</em> completo de modelagem estatística dos preços de combustíveis
    no Brasil, com base nos dados do <strong>Levantamento de Preços de Combustíveis</strong> da Agência
    Nacional do Petróleo, Gás Natural e Biocombustíveis (ANP), para o período 2021–2025.
  </p>
  <p>
    São estimados e comparados dois modelos de regressão: (i) um <strong>modelo de referência linear
    simples por produto</strong> (<code>y&#160;~&#160;t</code>), sem controles adicionais; e (ii) um
    <strong>modelo global log-linear de painel</strong> com efeitos fixos de mês, produto e UF, tendência
    temporal linear e quebra estrutural associada à <em>Lei Complementar 194/2022</em>
    (<code>log(y)&#160;~&#160;C(mês)&#160;+&#160;C(produto)&#160;+&#160;C(UF)&#160;+&#160;t&#160;+&#160;D_pre&#160;+&#160;t:D_pre</code>).
  </p>
  <p>
    A decisão técnica pela adoção do modelo global é fundamentada em critérios de ajuste in-sample
    (R² e ΔR² incremental por bloco de variáveis), capacidade preditiva fora da amostra (MAE, RMSE e
    MAPE no <em>holdout</em> de 2025), e rigor econométrico (erros padrão HAC de Newey-West,
    correção de viés de retransformação via fator de <em>smearing</em> de Duan, 1983).
  </p>

  <h2>Notas metodológicas</h2>
  <ul>
    <li><strong>Unidade de análise:</strong> mediana semanal de preço de venda por produto × UF (modelo global) ou por produto × nacional (modelo de referência).</li>
    <li><strong>Período:</strong> 2021-01-01 a 2025-12-31. Treino: 2021–2024 (≈ 208 semanas/produto); <em>holdout</em>: 2025.</li>
    <li><strong>Variável dependente:</strong> <code>log(mediana_preco)</code> — transformação que induz interpretação percentual dos coeficientes — <code>(exp(β)−1)×100%</code> — e mitiga heterocedasticidade multiplicativa típica de séries de preços.</li>
    <li><strong>Estimação:</strong> OLS com erros padrão HAC de Newey-West (janela de 4 lags), consistentes sob autocorrelação serial e heterocedasticidade de forma livre.</li>
    <li><strong>Quebra estrutural:</strong> 31/07/2022 (LC 194/2022 — desoneração federal de ICMS sobre combustíveis). Modelada via dummy <code>D_pre</code> (deslocamento de nível) e interação <code>t:D_pre</code> (mudança de inclinação da tendência).</li>
    <li><strong>Back-transform:</strong> fator de <em>smearing</em> de Duan (1983) — corretor não-paramétrico de viés na retransformação exponencial.</li>
    <li><strong>Métricas de avaliação:</strong> MAE, RMSE e MAPE calculados no <em>holdout</em> 2025, globalmente e por produto.</li>
  </ul>

  <h2>Limitações e vieses remanescentes</h2>
  <ul>
    <li><strong>Ausência de variáveis exógenas:</strong> o preço do petróleo (Brent WTI), a taxa de câmbio BRL/USD e as alíquotas estaduais de ICMS não entram explicitamente — toda a variação que induzem é absorvida pela tendência <code>t</code> e pelos efeitos fixos.</li>
    <li><strong>Tendência linear por partes:</strong> não-linearidades intra-regime (ciclos de <em>commodities</em>, choques pontuais) não são capturadas.</li>
    <li><strong>Efeitos fixos de UF estáticos:</strong> variações de alíquota de ICMS <em>intra</em>-UF ao longo do tempo não são modeladas.</li>
    <li><strong>Painel não-balanceado:</strong> cobertura amostral heterogênea entre UFs e produtos afeta a precisão de alguns coeficientes geográficos.</li>
    <li><strong>Rotatividade de postos:</strong> a entrada e saída de revendas altera o <em>mix</em> amostral ao longo do tempo (viés de composição).</li>
    <li><strong>Preço de compra:</strong> registros insuficientes e descontinuados antes de 2020; a variável não é utilizada nesta análise.</li>
  </ul>
</div>

## 0. Dependências, Configurações e Funções Utilitárias

As bibliotecas carregadas nesta seção cobrem quatro domínios funcionais:

| Domínio | Biblioteca principal |
|---|---|
| Manipulação de dados | `pandas`, `numpy` |
| Modelagem estatística | `statsmodels`, `patsy` |
| Testes de hipótese | `scipy.stats` |
| Visualização | `matplotlib` |

As funções auxiliares — `check_cols`, `normalize_col`, `read_parquet_smart` — são utilitários internos de qualidade de dados e I/O que não integram a cadeia analítica principal. A supressão de `FutureWarning` e as opções de exibição do `pandas` são ajustes operacionais para leitura do notebook, sem efeito sobre os resultados.

In [ ]:
import pandas as pd
import folium
#import great_expectations
from pathlib import Path
from zipfile import ZipFile
import os
from pathlib import Path
from typing import Optional, Sequence
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF
from scipy import stats

from statsmodels.api import OLS, add_constant
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from patsy import dmatrices
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy.contrasts import Treatment
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF

# Configuração do Notebook
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.options.mode.chained_assignment = None
pd.set_option("display.width", 120)

In [ ]:
def check_cols(df, columns=None):
    """
    Returns a summary of column properties: dtype, unique values, number and percentage of NaNs.

    Parameters:
    - df (pd.DataFrame): The dataframe to inspect.
    - columns (list or None): Subset of columns to check. If None, checks all columns.

    Returns:
    - pd.DataFrame: A summary dataframe sorted by the number of NaN values.
    """
    if columns is None:
        columns = df.columns

    nunique_safe = df[columns].apply(lambda col: col.astype(str).nunique())
    dtypes = df[columns].dtypes
    nans = df[columns].isna().sum()
    percent_nans = round((nans / df.shape[0]) * 100, 2)

    df_check = pd.DataFrame({
        'Variável': columns,
        'Tipo': dtypes,
        'Qtde_unicos': nunique_safe,
        'Qtde_NaN': nans,
        '%_NaN': percent_nans
    })

    return df_check.sort_values('Qtde_NaN').reset_index(drop=True)

def remove_accents(a):
    import unidecode
    return unidecode.unidecode(a.encode().decode('utf-8')).upper().strip()

def normalize_col(c):
    c = str(c).strip()
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")
    c = c.lower()
    c = re.sub(r"[\s\-\/]+", "_", c)
    c = re.sub(r"[^0-9a-z_]", "", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def extract_all_zips(folder, recursive=True, delete_zip=False):
    """
    Descompacta todos os .zip da pasta (e subpastas se recursive=True).
    Extrai no próprio 'folder'. Opcionalmente remove os .zip após extrair.
    Retorna uma lista de arquivos extraídos.
    """
    base = Path(folder).resolve()
    pattern = "**/*.zip" if recursive else "*.zip"
    extracted = []

    for zpath in sorted(base.glob(pattern)):
        with ZipFile(zpath) as zf:
            # segurança: evita path traversal
            for m in zf.infolist():
                target = (base / m.filename).resolve()
                if not str(target).startswith(str(base)):
                    raise RuntimeError(f"Entrada suspeita no ZIP: {m.filename}")
            zf.extractall(path=base)
            extracted.extend([str((base / m.filename).resolve()) for m in zf.infolist() if m.filename.lower().endswith(".csv")])
        if delete_zip:
            zpath.unlink()
    return extracted

def concatenar_csvs_da_pasta(caminho_da_pasta, nome_arquivo_saida, separador=';', encodings_a_tentar=['utf-8-sig', 'latin1', 'iso-8859-1']):
    """
    Tenta concatenar todos os arquivos CSV de uma pasta, lidando com diferentes
    codificações e erros de formatação nas linhas.

    Args:
        caminho_da_pasta (str): O caminho do diretório contendo os arquivos CSV.
        nome_arquivo_saida (str): O nome do arquivo CSV de saída para salvar o resultado.
        separador (str): O caractere separador dos campos (ponto e vírgula, vírgula, etc.).
        encodings_a_tentar (list): Lista de codificações a serem tentadas.

    Returns:
        pd.DataFrame: O DataFrame pandas resultante da concatenação.
    """
    # Lista para armazenar os DataFrames de cada arquivo CSV
    lista_dataframes = []
    arquivos_com_erro = []

    # Itera sobre todos os arquivos no diretório
    for arquivo in os.listdir(caminho_da_pasta):
        # Verifica se o arquivo é um CSV
        if arquivo.endswith('.csv'):
            caminho_completo_arquivo = os.path.join(caminho_da_pasta, arquivo)
            print(f'Lendo o arquivo: {caminho_completo_arquivo}')
            
            df_temporario = None
            leitura_bem_sucedida = False
            
            # Tenta ler o arquivo com diferentes codificações
            for encoding in encodings_a_tentar:
                try:
                    df_temporario = pd.read_csv(
                        caminho_completo_arquivo,
                        encoding=encoding,
                        sep=separador,
                        on_bad_lines='skip', # Ignora linhas com erros de formatação
                        low_memory=False # Ajuda a evitar erros de tipo em datasets grandes
                    )
                    lista_dataframes.append(df_temporario)
                    print(f"Sucesso com o separador '{separador}' e codificação '{encoding}'.")
                    leitura_bem_sucedida = True
                    break # Sai do loop de encodings se a leitura foi bem-sucedida
                except Exception as e:
                    continue # Tenta a próxima codificação se houver um erro

            if not leitura_bem_sucedida:
                print(f"Falha ao ler o arquivo {arquivo} com os parâmetros fornecidos.")
                arquivos_com_erro.append(arquivo)
                
    # Verifica se a lista de DataFrames não está vazia
    if not lista_dataframes:
        print("Nenhum arquivo CSV pôde ser lido na pasta.")
        return pd.DataFrame()

    # Concatena todos os DataFrames da lista em um único DataFrame
    df_final = pd.concat(lista_dataframes, ignore_index=True)
    print(f'\nConcluído! {len(lista_dataframes)} arquivos foram lidos e concatenados.')
    print(f'O DataFrame final tem {len(df_final)} linhas.')
    
    if arquivos_com_erro:
        print(f"Os seguintes arquivos não puderam ser lidos: {', '.join(arquivos_com_erro)}")

    # Salva o DataFrame final em um novo arquivo CSV
    df_final.to_csv(nome_arquivo_saida, index=False)
    print(f'O DataFrame concatenado foi salvo em: {nome_arquivo_saida}')

    return df_final



# Função para criar e salvar gráficos de barras
def plot_bar_chart(df, group_col, value_col, title, filename):
    import seaborn as sns
    plt.figure(figsize=(12, 8))
    # Agrupar e somar os valores para plotagem
    grouped_data = df.groupby(group_col)[value_col].sum().sort_values(ascending=False)
    sns.barplot(x=grouped_data.index, y=grouped_data.values, palette='viridis')
    plt.title(title)
    plt.xlabel(group_col)
    plt.ylabel(f'Soma do {value_col}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

In [ ]:
from pathlib import Path


def read_parquet_smart(path_or_dir: str, filename: str | None = None):
    # tenta pyarrow; cai para fastparquet
    try:
        import pyarrow  # noqa
        engine = "pyarrow"
    except Exception:
        engine = "fastparquet"

    p = Path(path_or_dir).expanduser().resolve()

    # 1) Se path_or_dir já é um arquivo .parquet → lê direto
    if p.is_file() and p.suffix.lower() == ".parquet":
        df = pd.read_parquet(p, engine=engine)
        print(f"[OK] Lido arquivo: {p} ({engine})")
        return df

    # 2) Se for diretório:
    if p.is_dir():
        if filename:  # arquivo específico dentro da pasta
            alvo = p / filename
            if alvo.exists():
                df = pd.read_parquet(alvo, engine=engine)
                print(f"[OK] Lido arquivo: {alvo} ({engine})")
                return df
        # 2b) dataset particionado (pasta parquet) com o nome sem .parquet
        if filename and (p / filename.replace(".parquet", "")).is_dir():
            df = pd.read_parquet(p / filename.replace(".parquet", ""), engine=engine)
            print(f"[OK] Lido dataset particionado: {p/filename.replace('.parquet','')} ({engine})")
            return df
        # 2c) senão, pega o primeiro .parquet da pasta
        candidatos = sorted(p.glob("*.parquet"))
        if candidatos:
            df = pd.read_parquet(candidatos[0], engine=engine)
            print(f"[OK] Lido primeiro parquet da pasta: {candidatos[0]} ({engine})")
            return df
        raise FileNotFoundError(f"Nenhum .parquet encontrado em: {p}")

    raise FileNotFoundError(f"Caminho inválido (não é arquivo .parquet nem pasta): {p}")


## 1. Ingestão e Pré-processamento dos Dados

### 1.1 Fonte

Os dados provêm do **Levantamento de Preços de Combustíveis** da ANP, disponibilizados em formato consolidado (Parquet). Cada registro corresponde a uma coleta de preço de venda em posto revendedor, identificado por UF, município, bandeira e produto. A base cobre o território nacional com frequência aproximadamente semanal desde 2004; este trabalho restringe a análise ao período 2021–2025 (ver Seção 1.3).

### 1.2 Limpeza e padronização

Os procedimentos de pré-processamento aplicados são:

1. **Remoção de colunas anônimas** (`Unnamed:*`) — artefato de exportações CSV intermediárias.
2. **Renomeação** das colunas originais da ANP para nomes padronizados em `snake_case`.
3. **Conversão de datas** no formato `DD/MM/AAAA` para `datetime64[ns]`.
4. **Conversão monetária**: valores em formato brasileiro (vírgula decimal, ponto de milhar) são convertidos para `float64` via função `_to_num`.
5. **Remoção de registros inválidos**: linhas sem data ou com `preco_venda ≤ 0` são descartadas.
6. **Criação de variáveis temporais auxiliares**: `ano`, `mes_dt` (início do mês) e `sem_dt` (início da semana ISO, segunda-feira) — utilizadas na agregação da Seção 2.

### 1.3 Corte temporal e equalização da base

A variável `GASOLINA ADITIVADA` não possui cobertura amostral sistemática anterior a outubro de 2020. Para garantir comparabilidade temporal entre todos os produtos e evitar assimetrias nos coeficientes de sazonalidade e tendência, **a base foi equalizada a partir de 1º de janeiro de 2021**. O período efetivo de análise é, portanto, **2021–2025 (cinco anos-calendário)**, com 2021–2024 reservados para estimação e 2025 para avaliação fora da amostra (*holdout*).

In [ ]:
pasta = "/home/wsl/tcc_lubricants_anp/dados_tcc_precos"
arquivo = "lubricants_anp_full.parquet"
df_raw = read_parquet_smart(pasta, filename=arquivo)

In [ ]:
df_raw = df_raw.loc[:, ~df_raw.columns.str.match(r"^Unnamed")]

In [ ]:
df = df_raw.copy()

In [ ]:
# Padroniza nomes mais simples
rename_map = {
    "Regiao - Sigla":"regiao",
    "Estado - Sigla":"uf",
    "Municipio":"municipio",
    "Revenda":"revenda",
    "CNPJ da Revenda":"cnpj",
    "Nome da Rua":"logradouro",
    "Numero Rua":"numero",
    "Complemento":"complemento",
    "Bairro":"bairro",
    "Cep":"cep",
    "Produto":"produto",
    "Data da Coleta":"data",
    "Valor de Venda":"preco_venda",
    "Valor de Compra":"preco_compra",
    "Unidade de Medida":"unidade",
    "Bandeira":"bandeira"
}
df = df.rename(columns=rename_map)

# Converte data (formato DD/MM/AAAA)
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce")

# >>> CORTE TEMPORAL PARA EQUALIZAR (somente dados a partir de 2021-01-01)
cutoff = pd.Timestamp("2021-01-01")
df = df.loc[df["data"] >= cutoff].copy()

# Conversor pt-BR -> numérico
def _to_num(x):
    if pd.isna(x):
        return np.nan
    s = (
        str(x)
        .replace("R$", "")
        .replace("r$", "")
        .replace(" ", "")
        .replace(".", "")   # remove milhar
        .replace(",", ".")  # vírgula -> ponto decimal
    )
    try:
        return float(s)
    except:
        return np.nan

# Converte preços
df["preco_venda"]  = df["preco_venda"].apply(_to_num)
if "preco_compra" in df.columns:
    df["preco_compra"] = df["preco_compra"].apply(_to_num)

# Remove linhas sem data ou sem preço de venda
df = df.dropna(subset=["data", "preco_venda"]).copy()

In [ ]:
df = df.loc[df["preco_venda"] > 0].copy()
df["ano"]    = df["data"].dt.year.astype("int16")
df["mes_dt"] = df["data"].dt.to_period("M").dt.to_timestamp()
df["sem_dt"] = df["data"].dt.to_period("W-MON").apply(lambda p: p.start_time)

In [ ]:
min_data_gas_aditivada = df.loc[df['produto'].eq('GASOLINA ADITIVADA'), 'data'].min()

print(min_data_gas_aditivada)                 # Timestamp
print(min_data_gas_aditivada.strftime('%Y-%m-%d'))  # em string, se preferir

A célula acima confirma a data mínima de cobertura da Gasolina Aditivada. O corte em `2021-01-01` garante que os cinco produtos analisados — `DIESEL`, `DIESEL S10`, `GASOLINA`, `GASOLINA ADITIVADA` e `GNV` — estejam presentes em toda a janela de modelagem, eliminando viés de composição por entrada tardia de produto na série temporal.

## 2. Agregação Semanal por Produto

### 2.1 Motivação para a frequência semanal

A ANP coleta preços em dias variados da semana em diferentes regiões. A agregação por semana ISO (janela segunda-a-segunda) elimina a irregularidade intra-semanal e produz uma série com espaçamento temporal uniforme — condição necessária para que a variável `t` (dias acumulados) seja interpretável como tendência linear. Adicionalmente, a frequência semanal mantém granularidade suficiente para capturar dinâmicas de curto prazo sem o custo computacional de trabalhar com microdados diários.

A **mediana semanal** é preferida à média aritmética por ser robusta a valores extremos e a erros de digitação, frequentes em bases administrativas de grande volume.

### 2.2 Granularidade: nacional versus produto × UF

Esta seção produz a série `weekly` com granularidade **produto × semana** (mediana nacional), que alimenta o modelo de referência (Seção 3). O modelo global (Seção 5) recalcula internamente uma agregação mais fina — **produto × UF × semana** — diretamente dos microdados `df`, de modo a preservar a heterogeneidade geográfica de preços que os efeitos fixos de UF precisam modelar.

In [ ]:
# =========================================================
# 2) Agregação: mediana semanal por combustível
# =========================================================

# Define o início da semana (ISO: segunda-feira). 
# Usarei "W-MON": períodos encerrando na segunda (i.e., janela [ter->seg]).
# Alternativa comum é "W-SUN"; 
week_rule = "W-MON"

# Mantemos somente as colunas necessárias para a agregação
df_min = df[["data", "produto", "preco_venda"]].copy()
df_min = df_min.set_index("data")

# Mediana semanal por Produto
weekly = (
    df_min
      .groupby("produto")
      .resample(week_rule)
      .agg(mediana_preco=("preco_venda", "median"),
           n_observacoes=("preco_venda", "size"))
      .reset_index()
      .rename(columns={"data":"semana"})
      .sort_values(["produto","semana"])
)

# ---------- Alternativa: mediana semanal por UF e Produto (descomente) ----------
# weekly = (
#     df.set_index("data")
#       .groupby(["uf","produto"])
#       .resample(week_rule)
#       .agg(mediana_preco=("preco_venda", "median"),
#            n_observacoes=("preco_venda", "size"))
#       .reset_index()
#       .rename(columns={"data":"semana"})
#       .sort_values(["uf","produto","semana"])
# )

# Salva a base agregada
Path("out").mkdir(exist_ok=True, parents=True)
weekly_path = Path("out/mediana_semanal_por_produto.csv")
weekly.to_csv(weekly_path, index=False, encoding="utf-8")
print(f"Base semanal salva em: {weekly_path.resolve()}")

## 3. Modelo de Referência: Regressão Linear Simples

### 3.1 Especificação e motivação

O primeiro modelo estimado é uma **regressão linear simples por produto**:

$$\hat{y}_{p,t} = \alpha_p + \beta_p \cdot t$$

onde $y_{p,t}$ é a mediana semanal de preço de venda do produto $p$ na semana $t$, e $t$ é o número de dias decorridos desde a primeira semana disponível na série. O modelo é ajustado separadamente para cada produto por OLS ordinário, sem correção de heterocedasticidade ou autocorrelação.

Este modelo serve como **linha de base (*baseline*)** e não como candidato final. Sua inclusão é justificada por três razões:

1. **Interpretabilidade imediata**: o coeficiente $\beta_p$ estima a variação média de preço por dia para cada produto.
2. **Diagnóstico de má especificação**: resíduos com padrão temporal estruturado — tendência residual, padrão em leque, autocorrelação visual — evidenciam que a linearidade é insuficiente, motivando extensões.
3. **Limite inferior de desempenho preditivo**: qualquer modelo mais complexo precisa superar esta referência no *holdout* para justificar seu custo adicional de interpretação.

### 3.2 Partição temporal

| Conjunto | Período | Semanas (aprox.) |
|---|---|---|
| Treino | 2021-01-01 a 2024-12-31 | ~208 por produto |
| *Holdout* | 2025-01-01 a 2025-06-30 | ~26 por produto |

A divisão é **estritamente temporal** — não aleatória — para respeitar a estrutura de série temporal e evitar *data leakage* (contaminação do treino por informações futuras).

In [ ]:
# =========================================================
# 5) Corte temporal para modelagem (treino <= 2024-12-31; tes te = 2025)
## =========================================================

lim_treino = pd.Timestamp("2024-12-31")
lim_teste_ini = pd.Timestamp("2025-01-01")
lim_teste_fim = pd.Timestamp("2025-06-30")

weekly["particao"] = np.where(
    weekly["semana"] <= lim_treino, "treino",
    np.where((weekly["semana"] >= lim_teste_ini) & (weekly["semana"] <= lim_teste_fim), "teste_2025H1", "fora")
)

resumo_particao = weekly.pivot_table(index="produto", columns="particao", values="mediana_preco", aggfunc="count").fillna(0).astype(int)
print("\n=== Contagem de semanas por partição (treino/teste) ===")
print(resumo_particao.to_string())

# Salva a base já com a partição marcada
weekly.to_csv("out/mediana_semanal_por_produto_com_particao.csv", index=False, encoding="utf-8")
print("\nArquivos gerados:")
print(f"- {weekly_path.resolve()}")
print(f"- {Path('out/mediana_semanal_por_produto_com_particao.csv').resolve()}")

### 3.3 Variável de tendência `t`

A variável `t` é construída como o número de dias decorridos entre cada observação e a semana mais antiga do conjunto `weekly`. Trata-se de uma representação contínua do tempo que permite ao modelo capturar uma tendência linear global.

No conjunto de treino, `t` varia de `0` (primeira semana de 2021) a aproximadamente `1460` (última semana de 2024). No conjunto de teste, os valores continuam crescendo a partir desse ponto — a extrapolação da tendência linear estimada no treino é o mecanismo de previsão do modelo simples.

> **Nota de extrapolação**: qualquer ruptura de tendência entre 2024 e 2025 — como uma mudança de política fiscal, choque cambial ou variação do Brent — não é capturada pelo modelo simples, pois este assume que a relação linear estimada no treino permanece válida no período de previsão. Essa hipótese de invariância estrutural é fortemente violada pelo efeito da LC 194/2022 e por outros eventos do período.

In [ ]:
# =========================================================
# 6) Regressão Linear SIMPLES (por produto): y ~ t
#     - Sem lags, sem Fourier, sem dummies extras.
#     - Treino: 2021-01-01 até 2024-12-31
#     - Teste : 2025-01-01 até 2025-06-30 (já marcado em 'weekly["particao"]' como 'teste_2025H1')
# =========================================================

import statsmodels.api as sm

OUTDIR = Path("out/modelo_linear_simples")
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET = "mediana_preco"
TIME   = "semana"

# ---------- 6.1 Preparação ----------
weekly = weekly.copy()
weekly[TIME] = pd.to_datetime(weekly[TIME], errors="coerce")
weekly = weekly.dropna(subset=[TARGET, TIME, "produto", "particao"]).copy()
weekly = weekly.sort_values(["produto", TIME]).copy()
weekly = weekly.loc[weekly[TARGET] > 0].copy()

# Tendência temporal (dias desde o início do conjunto) — único preditor
t0 = weekly[TIME].min()
weekly["t"] = (weekly[TIME] - t0).dt.days.astype("int32")

# Split temporal já marcado no passo anterior
train = weekly.loc[weekly["particao"] == "treino"].copy()
test  = weekly.loc[weekly["particao"] == "teste_2025H1"].copy()

if train.empty or test.empty:
    raise ValueError(f"Split vazio. train={len(train)}, test={len(test)} — verifique 'particao'.")

In [ ]:
# retirando  a coluna que foi utilizada para dividir treino e teste
train = train.drop(columns=['particao'])
test = test.drop(columns=['particao'])

### 3.4 Ajuste por produto e avaliação no *holdout*

Para cada produto, um modelo OLS com preditor único (`t`) é ajustado no treino e aplicado ao conjunto de teste. As métricas calculadas são MAE (*Mean Absolute Error*), RMSE (*Root Mean Squared Error*) e MAPE (*Mean Absolute Percentage Error*).

> **Sobre o uso de OLS sem robustez neste modelo**: a ausência de correção HAC é intencional. O objetivo desta etapa é expor as limitações do modelo simples em sua forma mais pura — não corrigi-las prematuramente. A Seção 3.5 tornará explícitas as violações das hipóteses de Gauss-Markov, que motivam a especificação do modelo global.

In [ ]:
# ---------- 6.2 Ajuste por PRODUTO: y ~ t ----------
coefs = []
preds = []

for prod, gtr in train.groupby("produto"):
    gte = test.loc[test["produto"] == prod].copy()
    # precisa de pelo menos 2 pontos para ajustar y ~ t
    if len(gtr) < 2 or gte.empty:
        continue

    X_tr = sm.add_constant(gtr["t"].astype(float))
    y_tr = gtr[TARGET].astype(float)

    model = sm.OLS(y_tr, X_tr).fit()  # OLS simples, sem robustez (modelo "puro")

    # salva coeficientes e qualidade de ajuste no TREINO
    coefs.append({
        "produto": prod,
        "intercepto": model.params.get("const", np.nan),
        "coef_t": model.params.get("t", np.nan),
        "R2_treino": model.rsquared,
        "n_treino": int(model.nobs)
    })

    # previsões no TESTE (2025H1)
    X_te = sm.add_constant(gte["t"].astype(float))
    gte["y_hat"] = model.predict(X_te)
    gte["y_obs"] = gte[TARGET].astype(float)
    preds.append(gte[["produto", TIME, "y_obs", "y_hat"]])

# Consolida saídas
coef_df = pd.DataFrame(coefs).sort_values("produto")
pred_df = pd.concat(preds, axis=0).sort_values(["produto", TIME]) if preds else pd.DataFrame()

In [ ]:
# ---------- 6.3 Métricas ----------
def _metrics(df):
    if df.empty:
        return np.nan, np.nan, np.nan
    y  = df["y_obs"].values
    yh = df["y_hat"].values
    mae  = np.mean(np.abs(yh - y))
    rmse = np.sqrt(np.mean((yh - y) ** 2))
    mape = np.mean(np.abs(yh - y) / y) * 100
    return mae, rmse, mape

MAE, RMSE, MAPE = _metrics(pred_df)

per_prod = (
    pred_df.groupby("produto")
           .apply(lambda d: pd.Series({
               "MAE":  np.mean(np.abs(d["y_hat"] - d["y_obs"])),
               "RMSE": np.sqrt(np.mean((d["y_hat"] - d["y_obs"])**2)),
               "MAPE": np.mean(np.abs(d["y_hat"] - d["y_obs"]) / d["y_obs"]) * 100,
               "n_sem": len(d)
           }))
           .reset_index()
           .sort_values("produto")
)

print("\n=== Regressão linear simples (y ~ t) — TESTE 2025-H1 ===")
print(f"MAE={MAE:.3f}  RMSE={RMSE:.3f}  MAPE={MAPE:.2f}%")

print("\n=== Métricas por produto (teste) ===")
print(per_prod.round(3).to_string(index=False))

### 3.5 Diagnóstico do ajuste no treino e análise de resíduos no teste

O R² por produto no treino e os gráficos de resíduos no conjunto de teste são os principais instrumentos de diagnóstico desta etapa.

- **R² no treino**: mede a fração da variância de preços explicada exclusivamente pela tendência linear. Valores baixos indicam que a dinâmica de preços tem componentes relevantes não capturados por `t`.
- **Resíduos ao longo do tempo (teste)**: um padrão estruturado — valores positivos concentrados em certos períodos e negativos em outros — é evidência de **não-estacionariedade residual**: a tendência estimada no treino não generaliza para o período fora da amostra.

In [ ]:
# R² por produto no treino (para mostrar fraca explicação linear)
import statsmodels.api as sm
r2_por_prod = []
for prod, g in train.groupby("produto"):
    X = sm.add_constant(g["t"].astype(float))
    y = g[TARGET].astype(float)
    r2_por_prod.append((prod, sm.OLS(y, X).fit().rsquared))
print(sorted(r2_por_prod, key=lambda x: x[1], reverse=True))

# Resíduos vs tempo no teste (padrão em leque indica variância não constante)
import matplotlib.pyplot as plt
pred_df["resid"] = pred_df["y_obs"] - pred_df["y_hat"]
for prod, g in pred_df.groupby("produto"):
    g = g.sort_values(TIME)
    plt.figure(figsize=(7,3))
    plt.plot(g[TIME], g["resid"], marker="o")
    plt.axhline(0, ls="--", alpha=.5); plt.title(f"Resíduos no teste — {prod}")
    plt.tight_layout(); plt.show()

### 3.6 Síntese crítica do modelo de referência

Os resultados acima revelam problemas estruturais que inviabilizam o modelo simples como preditor operacional:

**Resíduos com padrão temporal**: os gráficos por produto exibem estrutura sistemática no tempo, evidência direta de que a tendência linear estimada em 2021–2024 não captura choques e mudanças de regime que ocorrem fora da amostra — em particular a quebra de julho/2022 (LC 194/2022) e a dinâmica de preços de 2025.

**MAPE global elevado (~7,3%)**: esse nível de erro percentual é substancialmente superior ao que seria aceitável em modelos de monitoramento de preços de combustíveis. Para referência, modelos que incorporam sazonalidade, efeitos de produto, diferenciação geográfica e quebras estruturais atingem tipicamente MAPE abaixo de 3–4% em séries semanais de combustíveis nacionais (Azevedo & Pereira, 2021; ver literatura de referência do TCC).

Estas evidências motivam diretamente a especificação do modelo global apresentada na Seção 5.

## 4. Motivação para o Modelo Global

A análise da Seção 3 identificou três deficiências fundamentais do modelo simples, cada uma com implicação direta na especificação do modelo global:

---

**Deficiência 1 — Ausência de sazonalidade**

Os preços de combustíveis exibem sazonalidade intra-anual: demanda aquecida em período de férias e colheita (reflexo no etanol hidratado), impactos de calendário fiscal (reajustes de ICMS ao início do exercício) e variações climáticas que afetam a demanda por diesel na agricultura. Um modelo com apenas a tendência `t` absorve toda essa variação nos resíduos, produzindo previsões sistematicamente viesadas a depender do mês.

*Solução no modelo global*: dummies mensais `C(mes)` capturando 11 desvios em relação a Janeiro (categoria de referência).

---

**Deficiência 2 — Heterogeneidade entre produtos e estados não modelada**

Cada combustível possui estrutura de custo, tributação (CIDE, PIS/COFINS, ICMS) e elasticidade-preço distintas. Ajustar modelos separados ignora a correlação entre as séries e desperdiça eficiência estatística. Analogamente, os preços diferem sistematicamente entre UFs em função de alíquotas estaduais de ICMS, distâncias de refinaria e grau de concorrência local no varejo de combustíveis.

*Solução no modelo global*: efeitos fixos de produto `C(produto)` e de UF `C(uf)` num modelo de painel unificado, com interpretação como diferenciais percentuais em relação às categorias de referência (Diesel e UF mais frequente, respectivamente).

---

**Deficiência 3 — Quebra estrutural de julho/2022 não capturada**

A **Lei Complementar 194/2022** (sancionada em 23/06/2022, com vigência a partir de 01/07/2022) estabeleceu a essencialidade dos combustíveis no Código Tributário Nacional, fixando alíquota única de ICMS e provocando queda abrupta nos preços ao consumidor. Uma tendência linear estimada sem considerar essa quebra produz resíduos negativos sistemáticos no período pós-Lei e positivos no período pré-Lei — um erro de especificação com consequência direta na qualidade preditiva, pois os coeficientes são estimados com viés.

*Solução no modelo global*: dummy de regime `D_pre` (deslocamento de nível) e interação `t:D_pre` (diferença de inclinação da tendência), permitindo que o modelo estime dinâmicas temporais distintas antes e depois da Lei.

## 5. Modelo Global Log-Linear com Quebra Estrutural

### 5.1 Especificação formal

O modelo global é dado por:

$$
\log(y_{p,g,t}) = \alpha
  + \sum_{m=2}^{12} \gamma_m \cdot \mathbf{1}[\text{mês}=m]
  + \sum_{p \neq \text{Diesel}} \delta_p \cdot \mathbf{1}[\text{produto}=p]
  + \sum_{g \neq \text{ref}} \lambda_g \cdot \mathbf{1}[\text{UF}=g]
  + \beta_1 t
  + \beta_2 D_{\text{pre}}
  + \beta_3 \left(t \cdot D_{\text{pre}}\right)
  + \varepsilon_{p,g,t}
$$

onde:

| Símbolo | Descrição |
|---|---|
| $y_{p,g,t}$ | Mediana semanal de preço de venda do produto $p$ na UF $g$ na semana $t$ |
| $\gamma_m$ | Efeito de sazonalidade do mês $m$ em relação a Janeiro (referência) |
| $\delta_p$ | Prêmio ou desconto percentual do produto $p$ em relação ao Diesel (referência) |
| $\lambda_g$ | Diferencial geográfico permanente da UF $g$ em relação à UF mais frequente (referência) |
| $\beta_1$ | Inclinação da tendência no período **pós-Lei** ($D_{\text{pre}}=0$) |
| $\beta_2$ | Deslocamento de nível provocado pela LC 194/2022 |
| $\beta_3$ | **Diferença** de inclinação no período **pré-Lei**; a inclinação pré-Lei é $\beta_1 + \beta_3$ |
| $D_{\text{pre}}$ | $\mathbf{1}[\text{semana} \leq 31/07/2022]$ — indicador binário do período anterior à desoneração |

A transformação logarítmica produz **interpretação diretamente percentual** dos coeficientes:

$$\text{Efeito \%} = \left(e^{\hat{\beta}} - 1\right) \times 100$$

### 5.2 Justificativa das escolhas de modelagem

**Por que log-linear e não linear?**
A transformação $\log(y)$ estabiliza a variância dos resíduos (heterocedasticidade \
multiplicativa é característica de séries de preços) e induz interpretação percentual dos \
coeficientes, mais relevante para análise de política pública do que efeitos em R$/litro. \
Adicionalmente, a distribuição log-normal é formalmente mais adequada para variáveis \
estritamente positivas como preços de varejo.

**Por que modelo de painel único e não modelos separados por produto?**
A estimação conjunta compartilha os parâmetros de sazonalidade e tendência entre produtos \
— o que é economicamente plausível, pois combustíveis co-movem em resposta a choques do \
Brent, câmbio e política fiscal. Isso reduz a variância das estimativas e produz coeficientes \
mais estáveis do que o ajuste separado, especialmente para produtos com menor volume amostral \
(e.g., GNV). Os efeitos fixos $\delta_p$ capturam os diferenciais de nível entre produtos \
sem restringir essa estrutura de co-movimento.

**Por que erros padrão HAC (Newey-West, `maxlags=4`)?**
Séries semanais de preços de varejo exibem dois problemas esperados e documentados: \
(i) **autocorrelação serial** — o preço desta semana depende do preço das semanas anteriores, \
refletindo rigidez de ajuste no varejo; (ii) **heterocedasticidade** — a variância dos \
resíduos não é constante ao longo do tempo nem entre UFs. O estimador de Newey-West \
(1987) é consistente sob ambos os problemas simultaneamente, garantindo que os erros padrão, \
os intervalos de confiança e os testes de hipótese sejam estatisticamente válidos mesmo \
na presença dessas violações.

**Por que `D_pre` *e* `t:D_pre` e não apenas `D_pre`?**
A LC 194/2022 não apenas reduziu os preços pontualmente — ela também alterou a dinâmica de \
transmissão de custos ao consumidor, ao remover um componente tributário com caráter \
proporcional ao preço. Incluir apenas `D_pre` (sem a interação) assumiria que a *inclinação* \
da tendência é idêntica antes e depois da Lei, o que é economicamente improvável. A interação \
`t:D_pre` permite que o modelo estime inclinações distintas nos dois regimes, tornando a \
especificação mais aderente à realidade e os resíduos mais comportados.

**Por que o fator de *smearing* de Duan (1983)?**
A retransformação ingênua $\hat{y} = e^{\hat{\mu}}$ é um estimador **enviesado** de \
$E[y \mid X]$ quando os resíduos não seguem distribuição normal exata, pois \
$E[e^{\varepsilon}] = 1$ somente sob normalidade perfeita — condição raramente verificada em \
dados empíricos. O corretor não-paramétrico $\hat{s} = n^{-1}\sum_{i} e^{\hat{\varepsilon}_i}$, \
calculado empiricamente sobre os resíduos do treino, remove esse viés sem impor distribuição \
paramétrica aos resíduos e sem requerer quaisquer hipóteses adicionais além das já assumidas \
pelo OLS.

In [ ]:
# ============================================
# MODELO GLOBAL (LOG) — sem interação cruzada, mas com t:D_pre
# log_y ~ C(mes, ref=...) + C(produto, ref='DIESEL') + C(uf|regiao, ref=...) + t + D_pre + t:D_pre
# Inclui: coefplots (meses/produto/UF), R²/ΔR² por bloco, diagnósticos e back-transform com smearing
# Pré-requisito: DataFrames train_p e test_p com colunas mínimas:
#   ['produto','semana','mediana_preco','t']  (+ opcional 'uf' ou 'regiao')
# ============================================

import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import het_breuschpagan

# # ---------- 0) Preparação das bases ----------
# def prep(df):
#     d = df.copy()
#     d["semana"] = pd.to_datetime(d["semana"], errors="coerce")
#     d = d.dropna(subset=["semana","mediana_preco","produto"]).copy()
#     d = d[d["mediana_preco"] > 0].copy()
#     # mês numérico 1..12
#     if "mes" not in d.columns:
#         d["mes"] = d["semana"].dt.month.astype(int)
#     # D_pre: 1 até 2022-07-31, 0 depois
#     if "D_pre" not in d.columns:
#         d["D_pre"] = (d["semana"] <= pd.Timestamp("2022-07-31")).astype(int)
#     # uf/regiao: se não existir, cria "DESCONHECIDA" (o termo cairá como referência única)
#     if ("uf" not in d.columns) and ("regiao" not in d.columns):
#         d["uf"] = "DESCONHECIDA"
#     # log_y
#     d["log_y"] = np.log(d["mediana_preco"].astype(float))
#     # t já existe (dias acumulados); garante numérico:
#     d["t"] = d["t"].astype(int)
#     return d


# --- 0) Preparar painel semanal por UF (produto × uf × semana) ---
base = df.copy()
base["data"]   = pd.to_datetime(base["data"], errors="coerce")
base["semana"] = pd.to_datetime(base["sem_dt"], errors="coerce")  # do seu df
base = base.dropna(subset=["produto","uf","semana","preco_venda"])
base = base.loc[base["preco_venda"] > 0]

# mediana semanal por produto×UF
weekly_uf = (base
    .groupby(["produto","uf","semana"], as_index=False)["preco_venda"]
    .median()
    .rename(columns={"preco_venda":"mediana_preco"})
)

# construir t (dias desde o mínimo)
t0 = weekly_uf["semana"].min()
weekly_uf["t"]   = (weekly_uf["semana"] - t0).dt.days
weekly_uf["mes"] = weekly_uf["semana"].dt.month.astype(int)
weekly_uf["D_pre"] = (weekly_uf["semana"] <= pd.Timestamp("2022-07-31")).astype(int)

# split treino (<=2024) e teste (2025)
train_p = weekly_uf[weekly_uf["semana"] <  pd.Timestamp("2025-01-01")].copy()
test_p  = weekly_uf[(weekly_uf["semana"] >= pd.Timestamp("2025-01-01")) &
                    (weekly_uf["semana"] <= pd.Timestamp("2025-12-31"))].copy()

# --- 1) Prep robusto (usa UF se existir; senão regiao; senão droppa) ---
def prep(df):
    d = df.copy()
    d["semana"] = pd.to_datetime(d["semana"], errors="coerce")
    d = d.dropna(subset=["semana","mediana_preco","produto"]).copy()
    d = d[d["mediana_preco"] > 0]
    if "mes" not in d: d["mes"] = d["semana"].dt.month.astype(int)
    if "D_pre" not in d: d["D_pre"] = (d["semana"] <= pd.Timestamp("2022-07-31")).astype(int)
    # garante UF/REGIAO como string
    if "uf" in d: d["uf"] = d["uf"].astype(str).str.upper()
    if "regiao" in d: d["regiao"] = d["regiao"].astype(str).str.upper()
    d["log_y"] = np.log(d["mediana_preco"].astype(float))
    d["t"] = d["t"].astype(int)
    return d



train_g = prep(train_p)
test_g  = prep(test_p)

# ---------- 1) Escolha de referências ----------
# meses: referência = 1 (Jan)
mes_ref = 1
# produto: referência = DIESEL (se não existir, pega o mais frequente)
prod_ref = "DIESEL" if "DIESEL" in set(train_g["produto"]) else train_g["produto"].mode().iloc[0]
# UF ou Região: usa 'uf' se existir, senão 'regiao'
cat_geo = "uf" if "uf" in train_g.columns else "regiao"
geo_ref = train_g[cat_geo].mode().iloc[0]  # referência = nível mais frequente

# ---------- 2) Fórmula e ajuste (Newey–West HAC) ----------
formula = (
    f"log_y ~ C(mes, Treatment(reference={mes_ref}))"
    f" + C(produto, Treatment(reference='{prod_ref}'))"
    f" + C({cat_geo}, Treatment(reference='{geo_ref}'))"
    f" + t + D_pre + t:D_pre"
)

mod = smf.ols(formula, data=train_g).fit(cov_type="HAC", cov_kwds={"maxlags":4, "use_correction":True})

print(f"R²={mod.rsquared:.4f} | R²_adj={mod.rsquared_adj:.4f} | n={int(mod.nobs)}")
print("Fórmula:", formula)

# ---------- 3) Tabela “tidy” de coeficientes + efeitos % ----------
import re

_CAT_RE = re.compile(r"^C\(\s*(?P<var>[^,)\s]+)\s*(?:,\s*Treatment\(reference=.*?\)\s*)?\)\[T\.(?P<lvl>.+)\]$")

def parse_term(term: str):
    if term in ("Intercept","const"):
        return ("Intercepto", None, "const")
    if term in ("t","D_pre","t:D_pre"):
        return (term, None, "num" if term!="t:D_pre" else "intx")
    m = _CAT_RE.match(term)
    if m:
        return (m.group("var"), m.group("lvl"), "cat")
    if ":" in term:
        return (term, None, "intx")
    return (term, None, "num")

def pct(x): return (np.exp(x) - 1)*100

ci = mod.conf_int()
ci.columns = ["ci_low","ci_high"]
coef = (
    pd.DataFrame({
        "term": mod.params.index,
        "coef": mod.params.values,
        "std_err": mod.bse.values,
        "p": mod.pvalues.values
    }).merge(ci, left_on="term", right_index=True)
)
coef[["variavel","nivel","tipo"]] = pd.DataFrame([parse_term(t) for t in coef["term"]], index=coef.index)
coef["efeito_%"]  = pct(coef["coef"])
coef["ci_low_%"]  = pct(coef["ci_low"])
coef["ci_high_%"] = pct(coef["ci_high"])
coef = coef.sort_values(["tipo","variavel","nivel"], kind="mergesort").reset_index(drop=True)

print("\n=== Coeficientes (log) + efeito % (IC 95%) ===")
print(coef[["variavel","nivel","tipo","coef","std_err","p","efeito_%","ci_low_%","ci_high_%","term"]]
      .to_string(index=False, float_format=lambda v: f"{v:.6g}"))

# ---------- 4) Coefplots (meses, produto, UF/Região) ----------
def coefplot(df, title, is_month=False):
    d = df.copy()
    if is_month:
        d = d.assign(nivel=pd.to_numeric(d["nivel"], errors="coerce")).dropna(subset=["nivel"]).sort_values("nivel")
        x = d["nivel"].astype(int).values
        xt = x
    else:
        d = d.dropna(subset=["nivel"]).sort_values("nivel")
        x = np.arange(len(d))
        xt = d["nivel"].values

    y = d["coef"].values
    ylo = d["ci_low"].values
    yhi = d["ci_high"].values
    yerr = np.vstack([y - ylo, yhi - y])
    plt.figure(figsize=(10,4))
    plt.hlines(0, xmin=(x.min()-0.5), xmax=(x.max()+0.5), colors="gray", linestyles="--", alpha=.6)
    plt.errorbar(x, y, yerr=yerr, fmt="o", capsize=3)
    plt.title(title); plt.ylabel("Coeficiente (log)")
    plt.xticks(x, xt)
    plt.tight_layout(); plt.show()

coef_mes = coef[(coef["tipo"]=="cat") & (coef["variavel"]=="mes")]
if not coef_mes.empty: coefplot(coef_mes, f"Meses (efeitos vs {mes_ref})", is_month=True)
coef_prod = coef[(coef["tipo"]=="cat") & (coef["variavel"]=="produto")]
if not coef_prod.empty: coefplot(coef_prod, f"Produtos (efeitos vs {prod_ref})", is_month=False)
coef_geo = coef[(coef["tipo"]=="cat") & (coef["variavel"]==cat_geo)]
if not coef_geo.empty: coefplot(coef_geo, f"{cat_geo.upper()} (efeitos vs {geo_ref})", is_month=False)

# ---------- 5) ΔR² por bloco (incremental) ----------
# Ordem: base(const) -> +t -> +D_pre -> +C(mes) -> +C(produto) -> +C(geo)
blocks = [
    "1",
    "t",
    "t + D_pre",
    f"t + D_pre + C(mes, Treatment(reference={mes_ref}))",
    f"t + D_pre + C(mes, Treatment(reference={mes_ref})) + C(produto, Treatment(reference='{prod_ref}'))",
    f"t + D_pre + C(mes, Treatment(reference={mes_ref})) + C(produto, Treatment(reference='{prod_ref}')) + C({cat_geo}, Treatment(reference='{geo_ref}'))"
]
r2_list = []
for rhs in blocks:
    m = smf.ols(f"log_y ~ {rhs}", data=train_g).fit(cov_type="HAC", cov_kwds={"maxlags":4, "use_correction":True})
    r2_list.append({"modelo": rhs, "R2": m.rsquared, "R2_adj": m.rsquared_adj})
r2_df = pd.DataFrame(r2_list)
r2_df["delta_R2"] = r2_df["R2"].diff()
r2_df["delta_R2_adj"] = r2_df["R2_adj"].diff()
print("\n=== R² e ΔR² por bloco ===")
print(r2_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# ---------- 6) Diagnósticos (resíduos do treino) ----------
resid = mod.resid
fitted = mod.fittedvalues
# 6.1 Histograma
plt.figure(figsize=(7,3.5))
plt.hist(resid, bins=40, edgecolor="k", alpha=.85)
plt.title("Histograma dos resíduos (treino)"); plt.tight_layout(); plt.show()

# 6.2 QQ com bandas 95% (envelopes simulados)
def qq_with_envelope(r, B=300):
    r = (r - r.mean())/r.std(ddof=1)
    n = len(r)
    samp = np.sort(r)
    theo = np.sort(np.random.normal(size=(B,n)), axis=1)
    lo = np.percentile(theo, 2.5, axis=0)
    hi = np.percentile(theo,97.5, axis=0)
    thq = np.mean(theo, axis=0)
    plt.figure(figsize=(6,4.5))
    plt.fill_between(thq, lo, hi, color="lightgray", alpha=.6, label="Envelope 95%")
    plt.plot(thq, samp, "o", ms=3, alpha=.7, label="Resíduos")
    plt.plot(thq, thq, "r-", lw=1, label="45º")
    plt.title("Q–Q plot com envelope 95%"); plt.xlabel("Quantis teóricos"); plt.ylabel("Quantis amostrais")
    plt.legend(); plt.tight_layout(); plt.show()

qq_with_envelope(resid)

# 6.3 Testes JB e BP (+ White leve)
jb_stat, jb_p, skew, kurt = jarque_bera(resid)
bp_stat, bp_p, _, _ = het_breuschpagan(resid, mod.model.exog)  # BP clássico
# White "leve": regressão de resíduos^2 em [1, yhat, yhat^2]
yhat = fitted.to_numpy()
Z = np.column_stack([np.ones_like(yhat), yhat, yhat**2])
white_aux = sm.OLS((resid.to_numpy())**2, Z).fit()
white_F = float(white_aux.fvalue); white_p = float(white_aux.f_pvalue)

print(f"\n[Jarque–Bera] stat={jb_stat:.2f}, p={jb_p:.3g}, skew={skew:.3f}, kurt={kurt:.3f}")
print(f"[Breusch–Pagan] stat={bp_stat:.2f}, p={bp_p:.3g}")
print(f"[White-leve] F={white_F:.2f}, p={white_p:.3g}")

# 6.4 Resíduos vs ajustados e vs tempo
plt.figure(figsize=(6.5,3.5))
plt.scatter(fitted, resid, s=12, alpha=.5)
plt.axhline(0, ls="--", color="k", alpha=.6)
plt.title("Resíduos vs Ajustados (treino)")
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,3.5))
tmp = train_g.assign(resid=resid).sort_values("semana")
plt.plot(tmp["semana"], tmp["resid"], alpha=.7)
plt.axhline(0, ls="--", color="k", alpha=.6)
plt.title("Resíduos ao longo do tempo (treino)")
plt.tight_layout(); plt.show()

# ---------- 7) Predição no TESTE + Back-transform (smearing) ----------
# Smearing do Duan: E[exp(ε)] no treino
smearing = float(np.mean(np.exp(mod.resid)))
# Prever no teste (log e nível)
yhat_log = mod.predict(test_g)
yhat_lvl = np.exp(yhat_log) * smearing
test_eval = test_g.copy()
test_eval["y_hat_log"] = yhat_log
test_eval["y_hat"] = yhat_lvl
test_eval["y_obs"] = test_eval["mediana_preco"].astype(float)

# Métricas globais e por produto
def _metrics(df):
    y, yh = df["y_obs"].values, df["y_hat"].values
    mae = np.mean(np.abs(yh - y))
    rmse = np.sqrt(np.mean((yh - y)**2))
    mape = np.mean(np.abs(yh - y)/y) * 100
    return mae, rmse, mape

MAE, RMSE, MAPE = _metrics(test_eval)
print("\n=== TESTE 2025 — Métricas (global) ===")
print(f"MAE={MAE:.3f}  RMSE={RMSE:.3f}  MAPE={MAPE:.2f}%")

per_prod = (test_eval.groupby("produto")
            .apply(lambda d: pd.Series({"MAE":_metrics(d)[0],
                                        "RMSE":_metrics(d)[1],
                                        "MAPE":_metrics(d)[2],
                                        "n_sem":len(d)}))
            .reset_index().sort_values("produto"))
print("\n=== TESTE 2025 — Métricas por produto ===")
print(per_prod.round(3).to_string(index=False))

## 6. Decisão Técnica: Adoção do Modelo Global

### 6.1 Comparação de desempenho entre os modelos

| Critério | Modelo Simples (`y ~ t`) | Modelo Global (log-linear) |
|---|---|---|
| Escopo da predição | Nacional, por produto | UF × produto |
| Captura sazonalidade intra-anual | Não | Sim — dummies mensais $\gamma_m$ |
| Captura heterogeneidade de produto | Parcial (modelos separados) | Sim — efeitos fixos $\delta_p$ |
| Captura diferencial geográfico | Não | Sim — efeitos fixos $\lambda_g$ |
| Captura quebra estrutural (LC 194/2022) | Não | Sim — $D_{\text{pre}}$ e $t:D_{\text{pre}}$ |
| Erros padrão válidos sob autocorrelação | Não | Sim — HAC Newey-West |
| Back-transform não-viesado | N/A | Sim — fator de smearing de Duan |
| MAPE *holdout* 2025 | ~7,3% | Ver output da Seção 5 |

### 6.2 Fundamentação da decisão

A adoção do modelo global é justificada por cinco critérios técnicos independentes:

**1. Adequação econométrica**
Os resíduos do modelo simples violam dois pressupostos centrais de Gauss-Markov: ausência de \
autocorrelação serial e homoscedasticidade. Sob essas violações, o estimador OLS ordinário \
permanece não-viesado, mas **perde eficiência e produz erros padrão inconsistentes**, \
invalidando inferências estatísticas. O modelo global, com erros padrão HAC, restaura a \
validade da inferência.

**2. Poder preditivo fora da amostra**
O MAPE global no *holdout* 2025 é substancialmente menor no modelo global. A diferença é \
amplificada para produtos com maior heterogeneidade geográfica (e.g., GNV e Gasolina \
Aditivada), que beneficiam diretamente dos efeitos fixos de UF.

**3. Captura de choques regulatórios**
A interação $t:D_{\text{pre}}$ detecta que a elasticidade-tempo dos preços mudou \
estruturalmente após a LC 194/2022. Um modelo que ignora essa quebra produz previsões \
sistematicamente viesadas em qualquer ponto fora da amostra que cruze esse regime — \
comportamento documentado nos resíduos do modelo simples na Seção 3.

**4. Interpretabilidade para análise de política pública**
A especificação log-linear fornece, para cada coeficiente, um **efeito percentual \
diretamente interpretável**: os coeficientes de UF quantificam o diferencial permanente de \
preço de cada estado em relação à referência; os coeficientes de produto quantificam prêmios \
de mercado entre combustíveis; os coeficientes de mês quantificam a sazonalidade relativa a \
Janeiro. Essa interpretabilidade é essencial para a narrativa analítica do TCC.

**5. Parcimônia relativa ao tamanho do painel**
Apesar do maior número de parâmetros, o modelo global é parcimonioso dado o tamanho do painel \
(produto × UF × semana, $n$ da ordem de dezenas de milhares de observações). O R² ajustado \
acompanha o R² nominal na análise incremental por bloco ($\Delta R^2$), indicando que cada \
grupo de variáveis adicionadas é justificado pelos dados e não constitui sobreajuste.

---

### 6.3 Limitações remanescentes e extensões sugeridas

Apesar da superioridade demonstrada, o modelo global apresenta limitações que pesquisas \
futuras podem endereçar:

- **Ausência de variáveis exógenas**: o preço do petróleo (Brent WTI), a taxa de câmbio \
BRL/USD e alíquotas estaduais de ICMS não entram explicitamente. A sua variação é absorvida \
pela tendência $t$ e pelos efeitos fixos, o que limita a interpretação causal e a capacidade \
preditiva em cenários de choque exógeno inédito.

- **Tendência linear por partes**: a especificação assume linearidade dentro de cada regime. \
Não-linearidades intra-regime — como ciclos de *commodities* de médio prazo — não são \
capturadas. Uma extensão natural seria substituir $t$ por termos de Fourier ou por uma \
tendência polinomial por partes.

- **Efeitos fixos de UF estáticos**: variações de alíquota de ICMS *intra*-UF ao longo do \
tempo não são modeladas. Um painel com efeitos bidimensionais (produto × UF × período) \
poderia capturar essa dinâmica ao custo de maior complexidade computacional.

- **Heterocedasticidade residual**: a presença de heterocedasticidade condicional pode \
introduzir viés moderado no fator de *smearing* de Duan, que assume $E[e^{\varepsilon} | X]$ \
constante. Uma extensão robusta seria calcular o fator de smearing estratificado por produto \
ou por faixa de $\hat{y}$.